# FDE-Ops-AI: record real model runs on a free Kaggle GPU

This notebook serves an open model with vLLM on a Kaggle GPU. Then it runs the FDE-Ops-AI evaluations and records each model call in a cassette. After that, all runs replay from the cassettes on any computer, with no GPU and no cost.

**Settings before you start:**

1. Accelerator: **GPU T4 x2** (or P100). Internet: **on**.
2. The repository is private. Add a Kaggle secret with the name `GITHUB_TOKEN`. Use a fine-grained token with read access to this repository only. Alternative: upload a zip of the repository as a Kaggle dataset, and set `REPO_ZIP` below.
3. Run all cells. At the end, download `fdeops-recordings.zip` from the output panel.

In [ ]:
# Configuration. The model must support tool calls. The AWQ model fits on one T4 GPU.
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
TOOL_PARSER = "hermes"          # The tool-call format of Qwen2.5.
REPO = "github.com/YashhMahajan/FDE-Ops-AI.git"
REPO_ZIP = None                 # For example "/kaggle/input/fde-ops-ai/FDE-Ops-AI.zip"
BENCH_SPLIT = "test"            # The benchmark split. The demo also runs the dev split.
BASE_URL = "http://localhost:8000/v1"

In [ ]:
%%capture
!pip install -q "vllm>=0.6.3"

In [ ]:
# Start the vLLM server in the background, with tool calls on.
import subprocess, time, urllib.request

log = open("/kaggle/working/vllm.log", "w")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--quantization", "awq", "--dtype", "half",
     "--max-model-len", "16384", "--gpu-memory-utilization", "0.90",
     "--enable-auto-tool-choice", "--tool-call-parser", TOOL_PARSER,
     "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)

for i in range(180):
    try:
        urllib.request.urlopen(BASE_URL + "/models", timeout=2)
        print(f"vLLM is ready after {i * 5} s")
        break
    except Exception:
        time.sleep(5)
else:
    raise SystemExit("vLLM did not start. Read /kaggle/working/vllm.log")

In [ ]:
# Get the repository (private: a token from Kaggle secrets, or a zip from a Kaggle dataset).
import os, shutil, subprocess

def sh(*args):
    print("$", " ".join(args))
    subprocess.run(list(args), check=True)

os.chdir("/kaggle/working")
if REPO_ZIP:
    shutil.unpack_archive(REPO_ZIP, "/kaggle/working")
else:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1",
                    f"https://x-access-token:{token}@{REPO}", "FDE-Ops-AI"], check=True)
os.chdir("/kaggle/working/FDE-Ops-AI")
sh("pip", "install", "-q", "uv")
sh("uv", "sync", "--locked")

In [ ]:
# 1. The regression set (dev split, all cases) for CI replay.
import re
text = open("evals/regression.yaml").read()
text = re.sub(r"(?m)^(    provider: ).*$", lambda m: m.group(1) + "vllm", text)
text = re.sub(r"(?m)^(    model: ).*$", lambda m: m.group(1) + MODEL, text)
text = re.sub(r"(?m)^    limit: .*\n", "", text)
text = re.sub(r"(?m)^(    cassettes: .*)$", lambda m: m.group(1) + "\n    base_url: " + BASE_URL, text)
open("evals/regression.yaml", "w").write(text)
sh("uv", "run", "fdeops", "regress", "evals/regression.yaml", "--record")

In [ ]:
# 2. The benchmark: the production configuration on the test split, for each tenant, 3 trials.
for t in ["ledgerloop", "parcelpoint"]:
    sh("uv", "run", "fdeops", "dataset", "build", "--tenant", f"tenants/{t}", "--out", "datasets")
    sh("uv", "run", "fdeops", "run", "--tenant", f"tenants/{t}",
       "--agent", f"configs/agents/{t}-production.yaml",
       "--provider", "vllm", "--model", MODEL, "--base-url", BASE_URL,
       "--dataset", f"datasets/{t}/support-core@v1", "--split", BENCH_SPLIT, "--trials", "3",
       "--llm-mode", "record", "--cassettes", f"evals/cassettes/bench-{t}-{BENCH_SPLIT}.jsonl",
       "--workers", "4", "--out", "runs")

In [ ]:
# 3. The flagship demo with the real model, and the export for the web UI.
for t in ["ledgerloop", "parcelpoint"]:
    sh("uv", "run", "fdeops", "demo", "--tenant", f"tenants/{t}",
       "--agent", f"configs/agents/{t}-production.yaml",
       "--provider", "vllm", "--model", MODEL, "--base-url", BASE_URL,
       "--llm-mode", "record", "--cassettes", f"evals/cassettes/demo-{t}.jsonl",
       "--out", "demo-output", "--export", "web/public/demo")

In [ ]:
# Pack only the results (not the virtual environment). Put them in the same folders locally.
import zipfile, pathlib
keep = ["evals", "runs", "demo-output", "web/public/demo"]
with zipfile.ZipFile("/kaggle/working/fdeops-recordings.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for top in keep:
        for f in pathlib.Path(top).rglob("*"):
            if f.is_file():
                z.write(f)
server.terminate()
print("Download /kaggle/working/fdeops-recordings.zip from the output panel.")
